In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from jax.example_libraries import stax, optimizers
import matplotlib.pyplot as plt
import pyvista as pv
import pinns 
import datetime
import jax.scipy.optimize
import jax.flatten_util
import scipy
import scipy.optimize
import random

rnd_key = jax.random.PRNGKey(1234)
np.random.seed(14124)

In [ ]:
jax.config.update("jax_enable_x64", False)
print("Devices: ", jax.devices())
dev = jax.devices('gpu')[0] if jax.device_count()>1 and len(jax.devices('gpu'))>0 else jax.devices('cpu')[0]
dev = jax.devices('gpu')[0]
print(dev)


In [ ]:
def create_quarter_annulus(r_inner, r_outer, angle_start, angle_end):
    """
    Create a quarter annulus NURBS patch.
    
    Parameters:
    - r_inner: inner radius
    - r_outer: outer radius
    - angle_start: starting angle in radians
    - angle_end: ending angle in radians
    
    Returns:
    - PatchNURBS geometry
    """
    # Define B-spline bases (3D geometry with thin z-direction)
    basis_radial = pinns.functions.BSplineBasisJAX(np.array([-1, 0, 1]), 2)
    basis_angular = pinns.functions.BSplineBasisJAX(np.array([-1, 1]), 2)
    basis_z = pinns.functions.BSplineBasisJAX(np.array([-1, 1]), 1)  # Linear in z
    
    # Control points array [radial_points, angular_points, z_points, spatial_dim]
    pts = np.zeros([4, 3, 2, 3])
    weights = np.ones([4, 3, 2])
    
    # Calculate angle properties
    angle_mid = (angle_start + angle_end) / 2
    angle_span = angle_end - angle_start
    
    # For NURBS circular arc, we need special weighting
    # The middle control point weight for a circular arc
    a = np.pi/2 - angle_span/2
    w_mid = np.sin(a)
    
    # Radii for the 4 control points (quadratic in radial direction)
    rs = np.linspace(r_inner, r_outer, 4)
    
    # Define the angular control points (in local coordinates)
    # Start angle
    pts_angular_local = np.array([
        [np.cos(-angle_span/2), np.sin(-angle_span/2), 0],  # Start
        [1/np.sin(a), 0, 0],                                 # Middle (extended for NURBS)
        [np.cos(angle_span/2), np.sin(angle_span/2), 0]     # End
    ])
    
    # Create control points for each radius at z=0
    for i, r in enumerate(rs):
        pts[i, :, 0, :2] = r * pts_angular_local[:, :2]
        pts[i, :, 0, 2] = 0.0  # Bottom layer at z=0
    
    # Copy xy coordinates to z=1 layer and set z=1 for thickness
    pts[:, :, 1, :2] = pts[:, :, 0, :2]  # Same xy coordinates
    pts[:, :, 1, 2] = 1.0  # Top layer at z=1
    
    # Set the middle control point weights for NURBS circular representation
    weights[:, 1, :] = w_mid
    
    # Create the NURBS patch (3D geometry, dimension is 3)
    geom = pinns.geometry.PatchNURBS(
        [basis_radial, basis_angular, basis_z], pts, weights, 0, 3)
    
    # Rotate to the correct position
    rotation_angle = angle_mid
    geom.rotate((0, 0, rotation_angle))
    
    return geom

geoms = {}

# Patch 1: r in [1,2], angle [0, π/4]
geoms['patch_1'] = create_quarter_annulus(1.0, 2.0, 0, np.pi/4)

# Patch 2: r in [2,3], angle [0, π/4]
geoms['patch_2'] = create_quarter_annulus(2.0, 3.0, 0, np.pi/4)

# Patch 3: r in [2,3], angle [π/4, π/2]
geoms['patch_3'] = create_quarter_annulus(2.0, 3.0, np.pi/4, np.pi/2)

print("Created 3 geometry patches forming L-shape topology:")
print(f"  Patch 1: r ∈ [1,2], θ ∈ [0, π/4]")
print(f"  Patch 2: r ∈ [2,3], θ ∈ [0, π/4]")
print(f"  Patch 3: r ∈ [2,3], θ ∈ [π/4, π/2]")

In [ ]:
# Visualize the geometry patches
with jax.disable_jit():
    objects = [pinns.extras.plot(geoms[name], dict(), N=32) for name in geoms]

# Merge all objects for visualization
merged_obj = objects[0]
for i in range(1, len(objects)):
    merged_obj = merged_obj.merge(objects[i])

merged_obj.save('Lshape_test.vtk')

In [ ]:
# --- Determine patch connectivity ---
with jax.disable_jit(True):
    connectivity = pinns.geometry.match_patches(geoms, eps=1e-4, verbose=False)

names = list(geoms.keys())
for c in connectivity:
    print(c)

In [ ]:
def monom(x, axis, zero, one, alpha=1.0):
    return ((x[..., axis] - zero)[..., None] / (one - zero))**alpha

def create_nn_architectures(nl, n_blocks):
    """Build residual-block NN architectures for domain, interface, and edge."""
    acti = stax.elementwise(lambda x: jax.nn.leaky_relu(x)**2)
    w_init = jax.nn.initializers.glorot_normal()

    block_first = lambda nl: stax.serial(
        stax.FanInConcat(), stax.Dense(nl, W_init=w_init, b_init=jax.nn.initializers.zeros), acti)
    block = lambda nl: stax.serial(
        stax.FanOut(2),
        stax.parallel(
            stax.serial(stax.Dense(nl, W_init=w_init), acti, stax.Dense(nl, W_init=w_init), acti),
            stax.Identity),
        stax.FanInSum)

    nn_3d = stax.serial(block_first(nl), *[block(nl) for _ in range(n_blocks)], stax.Dense(1))
    nn_2d = stax.serial(block_first(nl), *[block(nl) for _ in range(n_blocks)], stax.Dense(1))
    nn_1d = stax.serial(block_first(nl), *[block(nl) for _ in range(max(1, n_blocks - 1))], stax.Dense(1))
    return nn_3d, nn_2d, nn_1d

In [ ]:
def create_model(nn_3d, nn_2d, geoms, names):
    """Factory that builds a Pinn model closing over the given NN architectures."""

    class Pinn(pinns.PINN):

        def __init__(self):
            self.weights = {}
            for n in names:
                self.weights[n] = nn_3d[0](rnd_key, ((-1, 3),))[1]
            self.weights['iface_12'] = nn_2d[0](rnd_key, ((-1, 2),))[1]
            self.weights['iface_23'] = nn_2d[0](rnd_key, ((-1, 2),))[1]

            def D_1(ws, x): return nn_3d[1](ws['patch_1'], (x,))
            def D_2(ws, x): return nn_3d[1](ws['patch_2'], (x,))
            def D_3(ws, x): return nn_3d[1](ws['patch_3'], (x,))

            def I_12(ws, x): return nn_2d[1](ws['iface_12'], (x[..., 1:3],))
            def I_23(ws, x): return nn_2d[1](ws['iface_23'], (x[..., 0:3:2],))

            def I_12_eff(ws, x): return I_12(ws, x) * monom(x, 1, -1, 1) * monom(x, 1, 1, -1)
            def I_23_eff(ws, x): return I_23(ws, x) * monom(x, 0, -1, 1) * monom(x, 0, 1, -1)

            def bubble_z(x): return monom(x, 2, -1, 1) * monom(x, 2, 1, -1)

            self.solutions = {
                'patch_1': lambda ws, x: (
                    D_1(ws, x) * monom(x, 0, -1, 1) * monom(x, 0, 1, -1) * monom(x, 1, -1, 1) * monom(x, 1, 1, -1)
                    + I_12_eff(ws, x) * monom(x, 0, -1, 1)
                ) * bubble_z(x),

                'patch_2': lambda ws, x: (
                    D_2(ws, x) * monom(x, 0, -1, 1) * monom(x, 0, 1, -1) * monom(x, 1, -1, 1) * monom(x, 1, 1, -1)
                    + I_12_eff(ws, x) * monom(x, 0, 1, -1)
                    + I_23_eff(ws, x) * monom(x, 1, -1, 1)
                ) * bubble_z(x),

                'patch_3': lambda ws, x: (
                    D_3(ws, x) * monom(x, 0, -1, 1) * monom(x, 0, 1, -1) * monom(x, 1, -1, 1) * monom(x, 1, 1, -1)
                    + I_23_eff(ws, x) * monom(x, 1, 1, -1)
                ) * bubble_z(x),
            }

            def f_target_phys(x):
                r = jnp.sqrt(x[..., 0:1]**2 + x[..., 1:2]**2)
                theta = jnp.arctan2(x[..., 1:2], x[..., 0:1])
                z = x[..., 2:3]
                f1 = jnp.sin(jnp.pi * (r - 1)) * jnp.sin(4 * theta) * jnp.sin(jnp.pi * z)
                f2 = jnp.sin(2 * jnp.pi * (r - 1)) * jnp.sin(8 * theta) * jnp.sin(2 * jnp.pi * z)
                return f1 + f2

            self.f_target_phys = f_target_phys

            def f_target_phys_scalar(x):
                r = jnp.sqrt(x[0]**2 + x[1]**2)
                theta = jnp.arctan2(x[1], x[0])
                z = x[2]
                f1 = jnp.sin(jnp.pi * (r - 1)) * jnp.sin(4 * theta) * jnp.sin(jnp.pi * z)
                f2 = jnp.sin(2 * jnp.pi * (r - 1)) * jnp.sin(8 * theta) * jnp.sin(2 * jnp.pi * z)
                return f1 + f2

            self.f_rhs = jax.vmap(lambda x: -jnp.trace(jax.hessian(f_target_phys_scalar)(x)))

            super(Pinn, self).__init__(geoms)

        def loss(self, training_parameters, points):
            jacs = [
                pinns.functions.jacobian(lambda x: self.solutions[n](training_parameters, x))(points[n].points_reference)
                for n in names
            ]
            jacs_x = [points[names[i]].jacobian_transformation(jacs[i]) for i in range(len(names))]

            total_loss = 0.0
            for i, n in enumerate(names):
                x_ref = points[n].points_reference
                x_phys = jax.lax.stop_gradient(geoms[n](x_ref))
                u_pred = self.solutions[n](training_parameters, x_ref)[..., 0]
                grad_u = jacs_x[i][..., 0, :]
                f_val = self.f_rhs(x_phys)
                energy_density = 0.5 * jnp.sum(grad_u**2, axis=-1) - f_val * u_pred
                total_loss += jnp.dot(energy_density, points[n].dx())

            return total_loss

    model = Pinn()
    return model, model.weights

In [ ]:
def train(model, weights, n_batches, n_points_batch, n_epochs, dev, grad_clip_norm=1.0, verbose=True):
    """Train the PINN model with Adam + piecewise-constant LR schedule.

    Returns (trained_params, loss_history, elapsed_seconds).
    """
    total_steps = n_batches * n_epochs

    lr = optimizers.piecewise_constant(
        [int(0.3 * total_steps), int(0.6 * total_steps), int(0.8 * total_steps), int(0.9 * total_steps)],
        [1e-4, 5e-5, 2e-5, 1e-5, 1e-6])
    opt_init, opt_update, get_params = optimizers.adam(lr)

    opt_state = opt_init(weights)
    params = get_params(opt_state)

    loss_and_grad = jax.value_and_grad(model.loss)

    def step(params, opt_state, key, step_no):
        points = model.points_MonteCarlo(n_points_batch, key)
        loss, grads = loss_and_grad(params, points)
        flat_grads, unravel = jax.flatten_util.ravel_pytree(grads)
        gnorm = jnp.sqrt(jnp.sum(flat_grads**2))
        scale = jnp.minimum(1.0, grad_clip_norm / (gnorm + 1e-12))
        grads = jax.tree_util.tree_map(lambda g: g * scale, grads)
        opt_state = opt_update(step_no, grads, opt_state)
        params = get_params(opt_state)
        return params, opt_state, loss

    step_jit = jax.jit(step, device=dev)

    # Warm-up / compile
    _ = step_jit(params, opt_state, rnd_key, 0)

    hist = []
    tme = datetime.datetime.now()

    for ep in range(n_epochs):
        losses = []
        for b in random.sample(range(n_batches), n_batches):
            step_no = ep * n_batches + b
            params, opt_state, loss = step_jit(
                params, opt_state, jax.random.PRNGKey(step_no), step_no)
            losses.append(float(loss))

        mean_loss = np.mean(losses)
        hist.extend(losses)
        if verbose:
            print(f'Epoch {ep+1:3d}/{n_epochs} - mean loss: {mean_loss:.6e}')

    elapsed = datetime.datetime.now() - tme
    if verbose:
        print(f'\nTraining complete in {elapsed}')

    model.weights = params
    return params, hist, elapsed.total_seconds()

In [ ]:
def plot_loss(hist, title='Adam Training Loss', savepath=None):
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.plot(hist)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=150)
    plt.show()

In [ ]:
def evaluate(model, params, geoms, names, n_points_eval=50000):
    """Compute relative L2 and H1 errors against the exact solution.

    Returns (l2_rel, h1_rel) as Python floats.
    """
    points = model.points_MonteCarlo(n_points_eval, jax.random.PRNGKey(9999))

    def f_exact_scalar(x):
        r = jnp.sqrt(x[0]**2 + x[1]**2)
        theta = jnp.arctan2(x[1], x[0])
        z = x[2]
        f1 = jnp.sin(jnp.pi * (r - 1)) * jnp.sin(4 * theta) * jnp.sin(jnp.pi * z)
        f2 = jnp.sin(2 * jnp.pi * (r - 1)) * jnp.sin(8 * theta) * jnp.sin(2 * jnp.pi * z)
        return f1 + f2

    grad_f_exact = jax.vmap(jax.grad(f_exact_scalar))

    jacs = [
        pinns.functions.jacobian(lambda x: model.solutions[n](params, x))(points[n].points_reference)
        for n in names
    ]
    jacs_x = [points[names[i]].jacobian_transformation(jacs[i]) for i in range(len(names))]

    l2_error_sq = 0.0
    h1_error_sq = 0.0
    l2_norm_sq = 0.0
    h1_norm_sq = 0.0

    for i, n in enumerate(names):
        x_ref = points[n].points_reference
        x_phys = geoms[n](x_ref)

        u_pred = model.solutions[n](params, x_ref)[..., 0]
        grad_u_pred = jacs_x[i][..., 0, :]

        u_exact = model.f_target_phys(x_phys)[..., 0]
        grad_u_exact = grad_f_exact(x_phys)

        w = points[n].dx()

        l2_error_sq += jnp.dot((u_pred - u_exact)**2, w)
        l2_norm_sq += jnp.dot(u_exact**2, w)

        h1_error_sq += jnp.dot(jnp.sum((grad_u_pred - grad_u_exact)**2, axis=-1), w)
        h1_norm_sq += jnp.dot(jnp.sum(grad_u_exact**2, axis=-1), w)

    l2_rel = float(jnp.sqrt(l2_error_sq / l2_norm_sq))
    h1_rel = float(jnp.sqrt(h1_error_sq / h1_norm_sq))
    return l2_rel, h1_rel


In [ ]:
def save_vtk(model, params, geoms, names, filename):
    """Export the predicted solution and point-wise error to a VTK file."""
    with jax.disable_jit():
        sol_dict = {n: lambda x, nn=n: model.solutions[nn](params, x) for n in names}
        err_dict = {n: lambda x, nn=n: model.solutions[nn](params, x) - model.f_target_phys(geoms[nn](x)) for n in names}
        objects = [pinns.extras.plot(geoms[n], {'u': sol_dict[n], 'error': err_dict[n]}, N=32) for n in names]

    obj_sol = objects[0]
    for i in range(1, len(objects)):
        obj_sol = obj_sol.merge(objects[i])

    obj_sol.save(filename)
    print(f"VTK saved to {filename}")

In [ ]:
import pandas as pd
from itertools import product

# --- Parametric study configuration ---
n_epochs = 100  # fixed across all runs for fair comparison

param_grid = {
    'nl':             [32, 64, 128],
    'n_blocks':       [2, 3],
    'n_batches':      [200, 500],
    'n_points_batch': [1000, 2000, 4000],
}

configs = list(product(
    param_grid['nl'],
    param_grid['n_blocks'],
    param_grid['n_batches'],
    param_grid['n_points_batch'],
))

print(f"Total configurations: {len(configs)}")

results = []
best_l2 = float('inf')
best_vtk_path = None

for run_idx, (nl, n_blocks, n_batches, n_points_batch) in enumerate(configs):
    tag = f"nl={nl}_blk={n_blocks}_nb={n_batches}_nppb={n_points_batch}"
    print(f"\n{'='*60}")
    print(f"Run {run_idx+1}/{len(configs)}: {tag}")
    print(f"{'='*60}")

    nn_3d, nn_2d, nn_1d = create_nn_architectures(nl, n_blocks)
    model, weights = create_model(nn_3d, nn_2d, geoms, names)

    params, hist, elapsed_s = train(
        model, weights, n_batches, n_points_batch, n_epochs, dev, verbose=False)

    l2_err, h1_err = evaluate(model, params, geoms, names)

    final_loss = float(np.mean(hist[-n_batches:]))
    print(f"  final_loss={final_loss:.4e}  L2={l2_err:.4e}  H1={h1_err:.4e}  time={elapsed_s:.1f}s")

    results.append({
        'nl': nl,
        'n_blocks': n_blocks,
        'n_batches': n_batches,
        'n_points_batch': n_points_batch,
        'total_points': n_batches * n_points_batch,
        'final_loss': final_loss,
        'L2_error': l2_err,
        'H1_error': h1_err,
        'time_s': elapsed_s,
    })

    if l2_err < best_l2:
        best_l2 = l2_err
        best_vtk_path = f'best_Lshape_L2_{best_l2:.4e}.vtk'
        save_vtk(model, params, geoms, names, best_vtk_path)
        print(f"  >>> New best L2! VTK saved to {best_vtk_path}")

print(f"\n{'='*60}")
print(f"Study complete. Best L2 error: {best_l2:.4e}")
print(f"Best VTK: {best_vtk_path}")

In [ ]:
# --- Results analysis ---
df = pd.DataFrame(results).sort_values('L2_error').reset_index(drop=True)
df.index += 1
df.index.name = 'rank'

print("=" * 80)
print("PARAMETRIC STUDY RESULTS  (sorted by L2 error, best first)")
print("=" * 80)
display(df)

print("\n\nSummary statistics:")
print("-" * 40)
summary = df[['L2_error', 'H1_error', 'final_loss', 'time_s']].describe()
display(summary)

# Per-parameter group means
print("\n\nMean L2 error by layer width (nl):")
display(df.groupby('nl')[['L2_error', 'H1_error']].mean())

print("\nMean L2 error by number of residual blocks:")
display(df.groupby('n_blocks')[['L2_error', 'H1_error']].mean())

print("\nMean L2 error by total points (n_batches * n_points_batch):")
display(df.groupby('total_points')[['L2_error', 'H1_error']].mean())

# Scatter plot: L2 vs H1, sized by network width, colored by total points
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
for nl_val in param_grid['nl']:
    sub = df[df['nl'] == nl_val]
    ax.scatter(sub['L2_error'], sub['H1_error'], label=f'nl={nl_val}', s=60, alpha=0.7)
ax.set_xlabel('Relative L2 error')
ax.set_ylabel('Relative H1 error')
ax.set_title('L2 vs H1 by layer width')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, which='both', alpha=0.3)

ax = axes[1]
for nb_val in param_grid['n_blocks']:
    sub = df[df['n_blocks'] == nb_val]
    ax.scatter(sub['total_points'], sub['L2_error'], label=f'n_blocks={nb_val}', s=60, alpha=0.7)
ax.set_xlabel('Total points per epoch')
ax.set_ylabel('Relative L2 error')
ax.set_title('L2 error vs total points')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, which='both', alpha=0.3)

ax = axes[2]
ax.scatter(df['time_s'], df['L2_error'], c=df['nl'], cmap='viridis', s=60, alpha=0.7)
ax.set_xlabel('Training time (s)')
ax.set_ylabel('Relative L2 error')
ax.set_title('L2 error vs training time')
ax.set_yscale('log')
ax.grid(True, which='both', alpha=0.3)
cbar = plt.colorbar(ax.collections[0], ax=ax)
cbar.set_label('nl (layer width)')

plt.tight_layout()
plt.savefig('parametric_study_results.png', dpi=150)
plt.show()